DSO459 Master Panel Construction

Merges NLP output (ai_scores.csv) with WRDS Compustat financials (financials_clean.csv).
Company universe was selected via yahoo finance API (DSO459_Project_Universe.ipynb).
ai_scores is filtered to the universe before merging with financials.
Further filtering to final regression sample happens in 459_Project_Regression.ipynb
where companies with fewer than 10 transcripts are excluded.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# Load company universe for reference — selected via Yahoo Finance API market cap ranking
universe = pd.read_csv("/content/drive/MyDrive/DSO 459/DSO459 Final Project /459 Earnings Calls Transcripts/universe.csv")
print("Company universe sourced via Yahoo Finance API:")
print(universe[["sector", "ticker"]].to_string())

# Load NLP output and financial data
ai_scores = pd.read_csv("/content/drive/MyDrive/DSO 459/DSO459 Final Project /459 Earnings Calls Transcripts/ai_scores.csv")
financials = pd.read_csv("/content/drive/MyDrive/DSO 459/DSO459 Final Project /459 Earnings Calls Transcripts/financials_clean.csv")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Company universe sourced via Yahoo Finance API:
                    sector ticker
0   Information Technology   NVDA
1   Communication Services  GOOGL
2   Information Technology   AAPL
3   Information Technology   MSFT
4   Information Technology   AVGO
5   Communication Services   META
6               Financials    JPM
7   Information Technology     MU
8   Information Technology    AMD
9               Financials      V
10  Information Technology   ORCL
11              Financials     MA
12             Industrials    CAT
13              Financials    BAC
14  Communication Services   NFLX
15  Information Technology   LRCX
16  Information Technology   CSCO
17  Information Technology   AMAT
18             Industrials     GE
19              Financials     MS
20             Industrials    GEV
21              Financials     GS
22              Financials    WFC
23     

In [ ]:
# Map Capital IQ company names to tickers
name_to_ticker = {
    "ATT Inc.": "T",
    "Advanced Micro Devices, Inc.": "AMD",
    "Alphabet Inc.": "GOOGL",
    "American Express Company": "AXP",
    "Apple Inc.": "AAPL",
    "Applied Materials, Inc.": "AMAT",
    "Bank of America Corporation": "BAC",
    "Broadcom Inc.": "AVGO",
    "Berkshire Hathaway Inc.": "BRK.B",
    "Caterpillar Inc.": "CAT",
    "Cisco Systems, Inc.": "CSCO",
    "Citigroup Inc.": "C",
    "Comcast Corporation": "CMCSA",
    "Deere  Company": "DE",
    "Discovery, Inc.": "WBD",
    "Electronic Arts Inc.": "EA",
    "GE Vernova Inc.": "GEV",
    "General Electric Company": "GE",
    "Honeywell International Inc.": "HON",
    "JPMorgan Chase  Co.": "JPM",
    "Lam Research Corporation": "LRCX",
    "Lockheed Martin Corporation": "LMT",
    "Mastercard Incorporated": "MA",
    "Meta Platforms, Inc.": "META",
    "Micron Technology, Inc.": "MU",
    "Microsoft Corporation": "MSFT",
    "Morgan Stanley": "MS",
    "NVIDIA Corporation": "NVDA",
    "Netflix, Inc.": "NFLX",
    "Oracle Corporation": "ORCL",
    "RTX Corporation": "RTX",
    "T-Mobile US, Inc.": "TMUS",
    "The Boeing Company": "BA",
    "The Goldman Sachs Group, Inc.": "GS",
    "The Walt Disney Company": "DIS",
    "Uber Technologies, Inc.": "UBER",
    "Union Pacific Corporation": "UNP",
    "Verizon Communications Inc.": "VZ",
    "Visa Inc.": "V",
    "Warner Bros. Discovery, Inc.": "WBD",
    "Wells Fargo  Company": "WFC"
}

# Add ticker column to AI scores
ai_scores["ticker"] = ai_scores["company"].map(name_to_ticker)

# Check for unmapped companies
unmapped = ai_scores[ai_scores["ticker"].isna()]["company"].unique()
print(f"\nUnmapped companies: {len(unmapped)}")
print(unmapped)
print(f"Mapped companies: {ai_scores['ticker'].notna().sum()} rows")


Unmapped companies: 0
[]
Mapped companies: 376 rows


In [ ]:
# Merge ai_scores with financials on ticker + year
# Companies outside WRDS coverage drop out here
merged = pd.merge(
    ai_scores,
    financials,
    left_on=["ticker", "year"],
    right_on=["Ticker", "Fiscal Year"],
    how="left"
)

print(f"\nTotal rows after merge: {len(merged)}")
print(f"Rows with matched financials: {merged['Revenue per Employee'].notna().sum()}")
print(f"Rows without matched financials: {merged['Revenue per Employee'].isna().sum()}")


Total rows after merge: 376
Rows with matched financials: 357
Rows without matched financials: 19


In [ ]:
# Find unmatched rows
unmatched = merged[merged["Revenue per Employee"].isna()]
print(f"Unmatched rows: {len(unmatched)}")
print(unmatched[["ticker", "company", "quarter", "year"]].to_string())

Unmatched rows: 19
    ticker                  company quarter  year
67      EA     Electronic Arts Inc.      Q3  2025
92      EA     Electronic Arts Inc.      Q4  2025
111   ORCL       Oracle Corporation      Q4  2025
118   ORCL       Oracle Corporation      Q2  2026
124   ORCL       Oracle Corporation      Q1  2025
170     MU  Micron Technology, Inc.      Q1  2026
186   NVDA       NVIDIA Corporation      Q3  2026
207   NVDA       NVIDIA Corporation      Q3  2025
314    GEV          GE Vernova Inc.      Q4  2024
319   NVDA       NVIDIA Corporation      Q2  2025
320     EA     Electronic Arts Inc.      Q2  2025
329     EA     Electronic Arts Inc.      Q1  2025
331   NVDA       NVIDIA Corporation      Q1  2025
337    GEV          GE Vernova Inc.      Q3  2024
349   NVDA       NVIDIA Corporation      Q4  2025
363     EA     Electronic Arts Inc.      Q1  2026
368    GEV          GE Vernova Inc.      Q1  2025
370    GEV          GE Vernova Inc.      Q3  2025
373   MSFT    Microsoft Corpora

In [ ]:
# Check what years we have for the problematic tickers
problem_tickers = ["EA", "ORCL", "MU", "NVDA", "GEV", "MSFT"]

for ticker in problem_tickers:
    years = financials[financials["Ticker"] == ticker]["Fiscal Year"].tolist()
    print(f"{ticker}: {years}")

# Unmatching results from fiscal year and calendar year inconsistencies

EA: [2020, 2021, 2022, 2023, 2024]
ORCL: [2020, 2021, 2022, 2023, 2024]
MU: [2021, 2022, 2023, 2024, 2025]
NVDA: [2020, 2021, 2022, 2023, 2024]
GEV: []
MSFT: [2021, 2022, 2023, 2024, 2025]


In [ ]:
# Drop unmatched rows
merged_clean = merged[merged["Revenue per Employee"].notna()].copy()
print(f"\nFinal dataset: {len(merged_clean)} rows")
print(f"Unique companies: {merged_clean['ticker'].nunique()}")
print(f"Years covered: {sorted(merged_clean['year'].unique())}")
print(f"Missing values: {merged_clean.isnull().sum().sum()}")


Final dataset: 357 rows
Unique companies: 38
Years covered: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Missing values: 0


In [ ]:
merged_clean.to_csv("/content/drive/MyDrive/DSO459/459 Earnings Calls Transcripts/master_panel.csv", index=False)